# BirdCLEF+ 2026 — Phase 5: Pseudo-Label Generation

This notebook does two things:

1. **Embed every 5-second window of every unlabeled soundscape file** with Perch v2.
   ~10,500 files × 12 windows = ~126,000 new embeddings (~750 MB on disk).
2. **Generate pseudo-labels** by running the 5-seed Phase 3.5 ensemble on those embeddings.
   Keep only species with predicted probability > **0.75** (high-confidence threshold).

**Output files** (in `embeddings/`):
- `unlabeled_ss_embeddings.npy` — shape `(~126_000, 1536)` float32
- `unlabeled_ss_index.csv` — columns: `filename, window_idx, start_sec, end_sec`
- `pseudo_label_probs.npy` — shape `(~126_000, 206)` float32, the raw ensemble probabilities
- `pseudo_labels.npy` — shape `(~126_000, 206)` float32, thresholded multi-hot labels

**Expected runtime**: ~60-90 min (embedding) + ~1 min (pseudo-labeling). Resumable.


## 1. Setup


In [1]:
import os, time
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import onnxruntime as ort
import torch
import torch.nn as nn
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data" / "birdclef-2026"
SS_AUDIO_DIR = DATA_DIR / "train_soundscapes"
PERCH_PATH   = PROJECT_ROOT / "data" / "perch" / "perch_v2_no_dft.onnx"
EMBED_DIR    = PROJECT_ROOT / "embeddings"
CKPT_DIR     = PROJECT_ROOT / "checkpoints"
EMBED_DIR.mkdir(exist_ok=True)

assert PERCH_PATH.exists()
assert (EMBED_DIR / "clip_embeddings.npy").exists(), "Run 03_perch_embed.ipynb first"
assert (CKPT_DIR / "model_v5_seed42.pt").exists(),    "Run Phase 3.5 training first"

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("DEVICE:", DEVICE)

SR        = 32000
CLIP_SEC  = 5
N_SAMPLES = SR * CLIP_SEC
N_WINDOWS = 60 // CLIP_SEC                  # 12 per 60s soundscape
EMBED_DIM = 1536


DEVICE: mps


## 2. Identify unlabeled soundscape files

Exclude any file already covered by `train_soundscapes_labels.csv` so we don't
re-embed those windows (we have them from `03_perch_embed.ipynb`).


In [2]:
labeled_files = set(pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")["filename"].unique())
all_files     = sorted(SS_AUDIO_DIR.glob("*.ogg"))
unlabeled_files = [p for p in all_files if p.name not in labeled_files]

print(f"All soundscape files:      {len(all_files):,}")
print(f"Labeled (excluded):        {len(labeled_files):,}")
print(f"Unlabeled (to be embedded): {len(unlabeled_files):,}")
print(f"Expected embeddings:       {len(unlabeled_files) * N_WINDOWS:,}")


All soundscape files:      10,658
Labeled (excluded):        66
Unlabeled (to be embedded): 10,592
Expected embeddings:       127,104


## 3. Load Perch + audio helpers


In [3]:
sess = ort.InferenceSession(str(PERCH_PATH), providers=["CPUExecutionProvider"])
INPUT_NAME    = sess.get_inputs()[0].name
EMBED_OUT_IDX = next(i for i, o in enumerate(sess.get_outputs()) if o.name == "embedding")


def load_audio(path):
    wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    return wav.astype(np.float32)


def file_to_12_chunks(path):
    """Return (12, 160000) — 12 non-overlapping 5s windows from a 60s file."""
    wav = load_audio(path)
    target = N_WINDOWS * N_SAMPLES   # 1,920,000
    if len(wav) < target:
        wav = np.pad(wav, (0, target - len(wav)))
    else:
        wav = wav[:target]
    return wav.reshape(N_WINDOWS, N_SAMPLES).astype(np.float32)


## 4. Extract embeddings for unlabeled soundscapes

Per-file we get 12 embeddings (one per 5s window). We batch all 12 in a single
Perch call, which is the natural unit. Total expected: ~10,500 files × 12 = ~126,000.

Resumable — saves every 200 files so an interrupt only loses a couple minutes.


In [ ]:
# Output paths
emb_path   = EMBED_DIR / "unlabeled_ss_embeddings.npy"
idx_path   = EMBED_DIR / "unlabeled_ss_index.csv"
prog_path  = EMBED_DIR / "unlabeled_ss_progress.txt"

N_TOTAL = len(unlabeled_files) * N_WINDOWS

# Resume support
if emb_path.exists() and prog_path.exists():
    embeddings = np.load(emb_path)
    start_file = int(prog_path.read_text().strip())
    print(f"Resuming from file {start_file}/{len(unlabeled_files)} ({start_file/len(unlabeled_files)*100:.1f}%)")
else:
    embeddings = np.zeros((N_TOTAL, EMBED_DIM), dtype=np.float32)
    start_file = 0

# Write the index CSV up-front (rows match embedding positions)
index_rows = []
for fi, p in enumerate(unlabeled_files):
    for w in range(N_WINDOWS):
        index_rows.append({
            "filename":   p.name,
            "window_idx": w,
            "start_sec":  w * CLIP_SEC,
            "end_sec":    (w + 1) * CLIP_SEC,
        })
pd.DataFrame(index_rows).to_csv(idx_path, index=False)

t0 = time.time()
pbar = tqdm(total=len(unlabeled_files), initial=start_file, desc="files")
for fi in range(start_file, len(unlabeled_files)):
    path = unlabeled_files[fi]
    try:
        chunks = file_to_12_chunks(path)                              # (12, 160000)
        outs = sess.run(None, {INPUT_NAME: chunks})
        emb12 = outs[EMBED_OUT_IDX]                                   # (12, 1536)
        embeddings[fi*N_WINDOWS : (fi+1)*N_WINDOWS] = emb12
    except Exception as e:
        print(f"[skip] {path.name}: {e}")
    pbar.update(1)
    if (fi + 1) % 200 == 0:
        np.save(emb_path, embeddings)
        prog_path.write_text(str(fi + 1))
pbar.close()

np.save(emb_path, embeddings)
prog_path.write_text(str(len(unlabeled_files)))
dt = time.time() - t0
print(f"\nDone. {(len(unlabeled_files) - start_file):,} files in {dt/60:.1f} min")
print(f"Embeddings: {embeddings.shape}  ({embeddings.nbytes/1e6:.0f} MB)")


files:   0%|          | 0/10592 [00:00<?, ?it/s]

## 5. Load the 5-seed ensemble heads

We use the same Phase 3.5 ensemble (`model_v5_seed{42..46}.pt`) to generate
pseudo-labels. Averaging 5 heads' logits gives smoother, less seed-biased
predictions than any single head.


In [ ]:
class PerchHead(nn.Module):
    def __init__(self, embed_dim=1536, hidden_dim=512, num_classes=234, dropout=0.3):
        super().__init__()
        self.norm  = nn.LayerNorm(embed_dim)
        self.drop1 = nn.Dropout(0.2)
        self.fc1   = nn.Linear(embed_dim, hidden_dim)
        self.act   = nn.ReLU(inplace=True)
        self.drop2 = nn.Dropout(dropout)
        self.fc2   = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.norm(x)
        x = self.drop1(x)
        x = self.act(self.fc1(x))
        x = self.drop2(x)
        return self.fc2(x)


SEEDS = [42, 43, 44, 45, 46]
heads = []
species, label_to_idx, NUM_CLASSES = None, None, None
for s in SEEDS:
    ckpt = torch.load(CKPT_DIR / f"model_v5_seed{s}.pt", map_location=DEVICE, weights_only=False)
    if species is None:
        species       = ckpt["species"]
        label_to_idx  = ckpt["label_to_idx"]
        NUM_CLASSES   = ckpt["num_classes"]
    h = PerchHead(EMBED_DIM, **ckpt["head_config"], num_classes=NUM_CLASSES).to(DEVICE)
    h.load_state_dict(ckpt["state_dict"])
    h.eval()
    heads.append(h)
print(f"Loaded {len(heads)} heads, NUM_CLASSES={NUM_CLASSES}")


## 6. Generate pseudo-label probabilities

For each unlabeled embedding, run all 5 heads, average their logits, sigmoid →
per-species probability. We batch this for speed.

Output: `(N_embeddings, 206)` float32 — one probability per species per window.


In [ ]:
probs_path = EMBED_DIR / "pseudo_label_probs.npy"

BATCH = 1024
probs = np.zeros((len(embeddings), NUM_CLASSES), dtype=np.float32)

t0 = time.time()
with torch.no_grad():
    for i in tqdm(range(0, len(embeddings), BATCH), desc="batches"):
        emb_batch = torch.from_numpy(embeddings[i:i+BATCH]).to(DEVICE)
        # Average logits across the 5 heads
        logits_avg = sum(h(emb_batch) for h in heads) / len(heads)
        probs[i:i+BATCH] = torch.sigmoid(logits_avg).cpu().numpy()

np.save(probs_path, probs)
print(f"\nDone in {time.time()-t0:.1f}s. Probs shape: {probs.shape}, range: [{probs.min():.3f}, {probs.max():.3f}]")


## 7. Apply threshold + sanity check the distribution

Threshold: **0.75** per species. Any species with `prob > 0.75` is set to 1 in
the pseudo-label vector; otherwise 0.

Sanity check what we got — if everything looks reasonable (most windows have
0-1 positives, distribution of species isn't wildly skewed), proceed.


In [ ]:
THRESHOLD = 0.75

pseudo_labels = (probs > THRESHOLD).astype(np.float32)
np.save(EMBED_DIR / "pseudo_labels.npy", pseudo_labels)

# Diagnostic stats
n_pos_per_window  = pseudo_labels.sum(axis=1)                # per-window positive count
n_pos_per_species = pseudo_labels.sum(axis=0)                # per-species positive count

print(f"Threshold: {THRESHOLD}")
print(f"Total windows: {len(pseudo_labels):,}")
print(f"Windows with 0 positives: {(n_pos_per_window == 0).sum():,}  ({(n_pos_per_window == 0).mean()*100:.1f}%)")
print(f"Windows with 1+ positives: {(n_pos_per_window > 0).sum():,}  ({(n_pos_per_window > 0).mean()*100:.1f}%)")
print(f"Mean positives per window (when > 0): {n_pos_per_window[n_pos_per_window > 0].mean():.2f}")
print()
print(f"Species coverage:")
print(f"  Species with 0 positives:     {(n_pos_per_species == 0).sum()}/{NUM_CLASSES}")
print(f"  Species with 1-9 positives:   {((n_pos_per_species > 0) & (n_pos_per_species < 10)).sum()}")
print(f"  Species with 10-99 positives: {((n_pos_per_species >= 10) & (n_pos_per_species < 100)).sum()}")
print(f"  Species with 100+ positives:  {(n_pos_per_species >= 100).sum()}")


## 8. Visualize: which species got pseudo-labeled the most?

Top 20 most-predicted species. Sanity check — if one species accounts for the
majority of pseudo-labels, the model has a bias problem and we'd want to
re-evaluate. A reasonable distribution has the top 20 species spread between
~100 and a few thousand positives, not 100k for one species.


In [ ]:
import matplotlib.pyplot as plt

# Top 20 species by pseudo-label count
top20 = np.argsort(-n_pos_per_species)[:20]
top20_species = [species[i] for i in top20]
top20_counts  = n_pos_per_species[top20]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(20), top20_counts)
ax.set_xticks(range(20))
ax.set_xticklabels(top20_species, rotation=45, ha="right")
ax.set_ylabel("Pseudo-label positives")
ax.set_title(f"Top 20 species by pseudo-label count (threshold={THRESHOLD})")
plt.tight_layout(); plt.show()


## What's next

When this finishes, share:

1. The "Windows with 1+ positives" percentage. Sensible range: 30-70%.
   - If much higher (90%+): threshold may be too low, too many noisy labels.
   - If much lower (<10%): threshold too high, almost no signal added.
2. The species coverage breakdown — most species should have at least a few positives.
3. The top-20 bar chart — should look like a long tail, not a single dominant species.

Then I'll rewrite `01_train.ipynb` to train on **clips + labeled soundscapes + pseudo-labeled soundscapes**. Expected val gain: +0.01 to +0.03. Expected LB gain: +0.03 to +0.05.
